<a href="https://colab.research.google.com/github/Fardous-bp/CNS-doped-Al-interconnect-alloy/blob/main/CNS_Al_12_5_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install matcalc

!pip install matgl
!pip install seekpath

!pip install crystal-toolkit

In [ ]:
import os
from pymatgen.core import Structure
my_cif_file = "CuNi2Sn.cif"

if os.path.exists(my_cif_file):
    my_structure = Structure.from_file(my_cif_file)
    print(f"SUCCESS: {my_cif_file} loaded successfully.")
    print(my_structure)
else:
    print(f"ERROR: File '{my_cif_file}' not found. Please check the name in your Colab files tab.")

In [ ]:
# The value 0.2 adds random noise (in Angstroms) to the atomic sites
cuni2sn_perturbed = my_structure.copy()
cuni2sn_perturbed.perturb(0.2)

# 2. Expand the lattice volume
# Multiplying by 1.2 increases the total cell volume by 20%
cuni2sn_perturbed.scale_lattice(my_structure.volume * 1.2)

# 3. View the results
print("Perturbed CuNi2Sn Structure:")
print(cuni2sn_perturbed)
cuni2sn_perturbed

In [ ]:
import matcalc
from matcalc.utils import UNIVERSAL_CALCULATORS

import pprint
pprint.pprint(list(UNIVERSAL_CALCULATORS))  # calculators that come with bundled with matgl

In [ ]:
from matcalc.utils import MODEL_ALIASES
pprint.pprint(MODEL_ALIASES)  # list all "aliased" models

In [ ]:
calculator_pbe = matcalc.load_fp("pbe")

In [ ]:
# Initialize the Relaxer exactly like the notebook
relax_calc = matcalc.RelaxCalc(
    calculator_pbe,
    optimizer="FIRE",
    relax_atoms=True,
    relax_cell=True,
)

# This should now complete in 1-3 minutes
print("Starting structural optimization...")
data = relax_calc.calc(cuni2sn_perturbed)

# Output results
print(f"Optimization Successful!")
print(f"Final Energy: {data['energy']:.4f} eV")
print(f"Final Optimized Volume: {data['final_structure'].volume:.2f} A^3")

In [ ]:
pprint.pprint(data)

In [ ]:
final_structure_pbe = data["final_structure"]
print(final_structure_pbe)
final_structure_pbe

In [ ]:
from matcalc import ElasticityCalc

# Use the 'final_structure_pbe' generated
multiplier_GPa = 160.2176621
elastic_calc = ElasticityCalc(calculator_pbe, relax_structure=False)
elastic_results = elastic_calc.calc(final_structure_pbe)

print(f"Bulk Modulus: {elastic_results['bulk_modulus_vrh'] * multiplier_GPa:.2f} GPa")
print(f"Shear Modulus: {elastic_results['shear_modulus_vrh'] * multiplier_GPa:.2f} GPa")

In [ ]:
from matcalc import PhononCalc
import matplotlib.pyplot as plt

# 1. Initialize for the pure structure
phonon_calc = PhononCalc(
    calculator_pbe,
    supercell_matrix=((2, 0, 0), (0, 2, 0), (0, 0, 2)),
    relax_structure=False
)

print("Calculating Phonon Data for Pure CuNi2Sn...")
pure_phonon_data = phonon_calc.calc(final_structure_pbe)

# 2. Access the Phonopy object
ph = pure_phonon_data["phonon"]

# 3. Trigger the DOS calculation (Fixes the AttributeError)
# We use a mesh of 20x20x20 for resolution
ph.run_mesh([20, 20, 20])
ph.run_total_dos()

# 4. Extract frequencies and densities
# get_total_dos_dict() returns a dictionary with 'frequency-points' and 'total-dos'
dos_dict = ph.get_total_dos_dict()
freqs = dos_dict['frequency_points']
densities = dos_dict['total_dos']

# 5. Visualization for Paper
plt.figure(figsize=(8, 5))
plt.plot(freqs, densities, color='blue', label='Pure CuNi2Sn')
plt.fill_between(freqs, densities, color='blue', alpha=0.2)
plt.ylim(0, 50)
plt.title("Phonon Density of States: Pure Baseline", fontsize=14)
plt.xlabel("Frequency (THz)", fontsize=12)
plt.ylabel("Density of States", fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# 1. Second 'Doped' version from the pure optimized structure
sac_al_12_5 = final_structure_pbe.copy()

# 2. Replace TWO sites with Aluminum to achieve 12.5% doping (2/16 atoms)
# We use site 12 (Sn) and site 0 (Cu) for a balanced distribution
sac_al_12_5.replace(12, "Al")
sac_al_12_5.replace(0, "Al")

print(f"Created 12.5% Al-doped structure: {sac_al_12_5.formula}")

# 3. Proceed to High-Precision Relaxation
print("Starting high-precision optimization for 12.5% Al...")
relax_results_12_5 = relax_calc.calc(sac_al_12_5)
opt_struct_12_5 = relax_results_12_5["final_structure"]

print(f"12.5% Optimized Energy: {relax_results_12_5['energy']:.4f} eV")

# 4. Calculate Elastic Moduli
elastic_results_12_5 = elastic_calc.calc(opt_struct_12_5)
multiplier_GPa = 160.2176621

bulk_12_5 = elastic_results_12_5['bulk_modulus_vrh'] * multiplier_GPa
shear_12_5 = elastic_results_12_5['shear_modulus_vrh'] * multiplier_GPa

print(f"--- 12.5% Al Doping Results ---")
print(f"Bulk Modulus: {bulk_12_5:.2f} GPa")
print(f"Shear Modulus: {shear_12_5:.2f} GPa")

In [ ]:
from matcalc import PhononCalc
import matplotlib.pyplot as plt
import numpy as np

# 1. Initialize Phonon Calculator for 12.5%
# Using the same supercell settings for a fair comparison
phonon_calc_12_5 = PhononCalc(
    calculator_pbe,
    supercell_matrix=((2, 0, 0), (0, 2, 0), (0, 0, 2)),
    relax_structure=False
)

print("Calculating Phonon Data for 12.5% Al-Doped structure...")
phonon_data_12_5 = phonon_calc_12_5.calc(opt_struct_12_5)
ph_12_5 = phonon_data_12_5["phonon"]

# 2. Run high-resolution DOS
ph_12_5.run_mesh([20, 20, 20])
ph_12_5.run_total_dos()

dos_dict_12_5 = ph_12_5.get_total_dos_dict()
freqs_12_5 = dos_dict_12_5['frequency_points']
densities_12_5 = dos_dict_12_5['total_dos']

# 3. Visualization
plt.figure(figsize=(8, 5))
plt.plot(freqs_12_5, densities_12_5, color='green', label='12.5% Al-Doped SAC')
plt.fill_between(freqs_12_5, densities_12_5, color='green', alpha=0.2)

# Set Y-axis range to 0-50 for consistent comparison
plt.ylim(0, 50)
plt.xlim(-1, 12)

plt.title("Phonon Density of States: 12.5% Al-Doped CuNi2Sn", fontsize=14)
plt.xlabel("Frequency (THz)", fontsize=12)
plt.ylabel("Density of States", fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


In [ ]:
# 4. Extract Thermodynamic Stability at Harsh Environment (473K)
ph_12_5.run_thermal_properties(t_step=5, t_max=600, t_min=0)
tp_12_5 = ph_12_5.get_thermal_properties_dict()

num_atoms = 16

# Use np to find the index closest to your target temperature (473K)
idx_473 = (np.abs(tp_12_5['temperatures'] - 473)).argmin()

# Apply the normalization factor to the target indexed outputs
normalized_free_energy = tp_12_5['free_energy'][idx_473] / num_atoms
normalized_entropy = tp_12_5['entropy'][idx_473] / num_atoms

print(f"--- 12.5% Al THERMODYNAMICS (NORMALIZED PER ATOM) ---")
print(f"Free Energy at 473K: {normalized_free_energy:.4f} kJ/mol·atom")
print(f"Entropy at 473K: {normalized_entropy:.4f} J/K/mol·atom")